# 🫀 Heart Disease Diagnosis: Exploratory Data Analysis & Gaussian Naive Bayes
**Author:** Om (Data Analysis & Gaussian Naive Bayes Lead)  
**Project:** CardioSense AI — Clinical Diagnostic Intelligence & Heart Disease Risk Prediction  
**Dataset:** UCI Cleveland Heart Disease Dataset (`data/heart_disease.csv`)  

---

## 📋 Om's Assigned Milestone Checklist:
- [x] **1. Exploratory Data Analysis (EDA) & Summary Statistics**
  - Inspect data types, nulls, and distributions via `df.info()` and `df.describe()`
  - **Graph 1**: Patient Age Distribution (`sns.histplot` + KDE, Mean, Median)
  - **Graph 2**: Heart Disease Occurrence vs Age (`sns.boxplot` & strip distribution)
  - **Graph 3**: Serum Cholesterol Distribution with 200 mg/dl clinical risk threshold
  - **Graph 4**: Disease Target Class Distribution (`sns.countplot` with percentages)
- [x] **2. Machine Learning: Gaussian Naive Bayes**
  - Stratified 80/20 train-test split aligned with central preprocessing pipeline
  - Scale continuous features via pre-fitted `StandardScaler` (`models/scaler.pkl`)
  - Implement, train, and test `GaussianNB` classifier
  - Compute **Accuracy**, **Precision**, **Recall (Sensitivity)**, **F1-Score**, **ROC-AUC**
  - Generate Confusion Matrix Display & ROC Curve
  - Save trained artifact to `models/naive_bayes.pkl`
- [x] **3. Clinical & Theoretical Documentation**
  - Clinical features and biomarker metadata dictionary
  - Key biological findings and diagnostic patterns from EDA
  - Mathematical formulation of Bayes' Theorem, conditional independence, and Gaussian likelihood

In [1]:
# 1. Import Libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve
)
import joblib

# Set cohesive clinical styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["font.size"] = 10

print("[+] All data science, visualization, and ML libraries loaded successfully!")

[+] All data science, visualization, and ML libraries loaded successfully!


## 2. Load Preprocessed Dataset & Handle Missing Values
We load the clinical cohort data and ensure any missing biomarker values (e.g. `ca`, `thal`) are cleanly imputed using column medians, perfectly matching Rohit's central preprocessing pipeline.

In [2]:
# Robust path resolution: works both when run inside notebooks/ or workspace root
data_path = "../data/heart_disease.csv" if os.path.exists("../data/heart_disease.csv") else "data/heart_disease.csv"

df = pd.read_csv(data_path)
print(f"[+] Successfully loaded dataset: {data_path}")
print(f"[+] Dataset dimensions: {df.shape[0]} patient records x {df.shape[1]} clinical features\n")

# Check for missing values and impute with column median
null_counts = df.isnull().sum()
if null_counts.sum() > 0:
    print("[!] Missing values detected; performing median imputation (matching Rohit's pipeline):")
    for col, count in null_counts[null_counts > 0].items():
        med_val = df[col].median()
        df[col] = df[col].fillna(med_val)
        print(f"    - {col}: {count} missing values -> Imputed with median ({med_val})")
else:
    print("[+] No missing values detected in dataset.")

df.head()

[+] Successfully loaded dataset: ../data/heart_disease.csv
[+] Dataset dimensions: 303 patient records x 14 clinical features

[!] Missing values detected; performing median imputation (matching Rohit's pipeline):
    - ca: 4 missing values -> Imputed with median (0.0)
    - thal: 2 missing values -> Imputed with median (3.0)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


## 3. Understand Columns & Summary Statistics
Inspect data types, memory usage, non-null counts, and descriptive statistical metrics (central tendency and dispersion).

In [3]:
# Column information & data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    float64
 1   sex       303 non-null    float64
 2   cp        303 non-null    float64
 3   trestbps  303 non-null    float64
 4   chol      303 non-null    float64
 5   fbs       303 non-null    float64
 6   restecg   303 non-null    float64
 7   thalach   303 non-null    float64
 8   exang     303 non-null    float64
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    float64
 11  ca        303 non-null    float64
 12  thal      303 non-null    float64
 13  target    303 non-null    int64  
dtypes: float64(13), int64(1)
memory usage: 33.3 KB


In [4]:
# Statistical summary across all 13 clinical biomarkers + target
df.describe().T[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]]

,count,mean,std,min,25%,50%,75%,max
age,303.0,54.438944,9.038662,29.0,48.0,56.0,61.0,77.0
sex,303.0,0.679868,0.467299,0.0,0.0,1.0,1.0,1.0
cp,303.0,3.158416,0.960126,1.0,3.0,3.0,4.0,4.0
trestbps,303.0,131.689769,17.599748,94.0,120.0,130.0,140.0,200.0
chol,303.0,246.693069,51.776918,126.0,211.0,241.0,275.0,564.0
fbs,303.0,0.148515,0.356198,0.0,0.0,0.0,0.0,1.0
restecg,303.0,0.990099,0.994971,0.0,0.0,1.0,2.0,2.0
thalach,303.0,149.607261,22.875003,71.0,133.5,153.0,166.0,202.0
exang,303.0,0.326733,0.469794,0.0,0.0,0.0,1.0,1.0
oldpeak,303.0,1.039604,1.161075,0.0,0.0,0.8,1.6,6.2


## 4. Exploratory Data Analysis (EDA Graphs)
We generate the 4 required exploratory graphs highlighting patient demographics, disease prevalence across age groups, serum cholesterol threshold adherence, and diagnostic class balance.

### Graph 1: Patient Age Distribution
Analyzes the age distribution across the 303 patients in the Cleveland clinic cohort with overlaid kernel density estimation (KDE), mean (54.4 years), and median (56.0 years).

In [5]:
# Graph 1: Patient Age Distribution
plt.figure(figsize=(9, 5))
sns.histplot(df['age'], kde=True, bins=20, color='#2b6cb0', edgecolor='white', alpha=0.75)

mean_age = df['age'].mean()
median_age = df['age'].median()
plt.axvline(mean_age, color='#c53030', linestyle='--', linewidth=2, label=f'Mean Age: {mean_age:.1f} yrs')
plt.axvline(median_age, color='#2f855a', linestyle=':', linewidth=2, label=f'Median Age: {median_age:.1f} yrs')

plt.title('Patient Age Distribution (Cleveland Cohort)', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Patient Age (years)', fontsize=11)
plt.ylabel('Patient Count', fontsize=11)
plt.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()

### Graph 2: Heart Disease Occurrence vs Patient Age
Compares age distributions between healthy individuals (Class 0) and patients with confirmed heart disease (Class 1).

In [6]:
# Graph 2: Heart Disease vs Patient Age (Boxplot with strip overlay)
plt.figure(figsize=(9, 5))
binary_target = (df['target'] > 0).astype(int)
plot_df = df.copy()
plot_df['binary_target'] = binary_target

palette = {0: '#3182ce', 1: '#e53e3e'}
sns.boxplot(x='binary_target', y='age', data=plot_df, palette=palette, width=0.45, boxprops=dict(alpha=0.85))
sns.stripplot(x='binary_target', y='age', data=plot_df, color='black', alpha=0.30, jitter=0.2, size=5)

med_healthy = plot_df[plot_df['binary_target'] == 0]['age'].median()
med_disease = plot_df[plot_df['binary_target'] == 1]['age'].median()
plt.text(0, med_healthy + 1.2, f'Median: {med_healthy:.0f} yrs', ha='center', color='darkblue', fontweight='bold')
plt.text(1, med_disease + 1.2, f'Median: {med_disease:.0f} yrs', ha='center', color='darkred', fontweight='bold')

plt.title('Heart Disease Diagnosis vs Patient Age', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Diagnostic Status', fontsize=11)
plt.ylabel('Patient Age (years)', fontsize=11)
plt.xticks([0, 1], ['Healthy / No Disease (0)', 'Heart Disease Present (1)'], fontsize=11)
plt.tight_layout()
plt.show()

### Graph 3: Serum Cholesterol Distribution
Examines total cholesterol levels with a red reference line indicating the clinical boundary for borderline high cholesterol ($200\text{ mg/dl}$).

In [7]:
# Graph 3: Serum Cholesterol Distribution with Clinical Reference Line
plt.figure(figsize=(9, 5))
sns.histplot(df['chol'], kde=True, bins=25, color='#319795', edgecolor='white', alpha=0.75)

# 200 mg/dl clinical guideline threshold
plt.axvline(x=200, color='#e53e3e', linestyle='--', linewidth=2.2, label='Desirable Threshold (200 mg/dl)')

pct_elevated = (df['chol'] > 200).mean() * 100
plt.text(210, plt.ylim()[1] * 0.85, f'{pct_elevated:.1f}% Patients > 200 mg/dl\n(Elevated Hyperlipidemia Risk)',
         color='#9b2c2c', fontsize=10, fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#fff5f5', edgecolor='#feb2b2'))

plt.title('Serum Cholesterol Distribution (mg/dl)', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Serum Cholesterol (mg/dl)', fontsize=11)
plt.ylabel('Patient Count', fontsize=11)
plt.legend(loc='upper right', frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()

### Graph 4: Target Class Distribution
Evaluates class balance between healthy patients ($N=164, 54.1\%$) and patients with heart disease ($N=139, 45.9\%$).

In [8]:
# Graph 4: Heart Disease Target Class Balance
plt.figure(figsize=(7, 5))
target_binary = (df['target'] > 0).astype(int)
ax = sns.countplot(x=target_binary, palette=['#4299e1', '#f56565'], edgecolor='white')

total = len(df)
for p in ax.patches:
    count = int(p.get_height())
    pct = (count / total) * 100
    ax.annotate(f'{count} ({pct:.1f}%)',
                (p.get_x() + p.get_width() / 2., count / 2),
                ha='center', va='center', fontsize=11, color='white', fontweight='bold')

plt.title('Diagnostic Target Class Balance', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Diagnostic Classification', fontsize=11)
plt.ylabel('Patient Count', fontsize=11)
plt.xticks([0, 1], ['No Disease (0)', 'Heart Disease Present (1)'], fontsize=11)
plt.tight_layout()
plt.show()

## 5. Machine Learning: Gaussian Naive Bayes Implementation
We prepare the stratified 80/20 train-test split, standardize clinical features via `StandardScaler`, fit `GaussianNB`, and compute key healthcare diagnostic metrics (**Accuracy**, **Precision**, **Recall/Sensitivity**, **F1-Score**, and **ROC-AUC**).

In [9]:
# 1. Features and Target separation
X = df.drop(columns=['target'])
y = (df['target'] > 0).astype(int)

# 2. Stratified 80/20 train-test split (random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"[+] Stratified 80/20 Split: {X_train.shape[0]} Training samples | {X_test.shape[0]} Testing samples")

# 3. Load pre-fitted StandardScaler from models/scaler.pkl (prepared by Rohit)
scaler_path = "../models/scaler.pkl" if os.path.exists("../models/scaler.pkl") else "models/scaler.pkl"
scaler = joblib.load(scaler_path)
print(f"[+] Loaded pre-fitted StandardScaler from: {scaler_path}")

X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

# 4. Train Gaussian Naive Bayes
gnb = GaussianNB()
gnb.fit(X_train_scaled, y_train)
print("[+] Gaussian Naive Bayes model successfully trained!")

# 5. Evaluate on test cohort
y_train_pred = gnb.predict(X_train_scaled)
y_test_pred = gnb.predict(X_test_scaled)
y_test_proba = gnb.predict_proba(X_test_scaled)[:, 1]

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)
prec = precision_score(y_test, y_test_pred, zero_division=0)
rec = recall_score(y_test, y_test_pred, zero_division=0)
f1 = f1_score(y_test, y_test_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_test_proba)

print("\n" + "=" * 60)
print("GAUSSIAN NAIVE BAYES BENCHMARK PERFORMANCE METRICS")
print("=" * 60)
print(f"Train Accuracy:        {train_acc * 100:.2f}%")
print(f"Test Accuracy:         {test_acc * 100:.2f}%")
print(f"Precision:             {prec * 100:.2f}%")
print(f"Recall (Sensitivity):  {rec * 100:.2f}%")
print(f"F1-Score:              {f1 * 100:.2f}%")
print(f"ROC-AUC Score:         {roc_auc:.4f}")
print("=" * 60)

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=['No Disease', 'Heart Disease']))

[+] Stratified 80/20 Split: 242 Training samples | 61 Testing samples
[+] Loaded pre-fitted StandardScaler from: ../models/scaler.pkl
[+] Gaussian Naive Bayes model successfully trained!

GAUSSIAN NAIVE BAYES BENCHMARK PERFORMANCE METRICS
Train Accuracy:        83.47%
Test Accuracy:         85.25%
Precision:             80.65%
Recall (Sensitivity):  89.29%
F1-Score:              84.75%
ROC-AUC Score:         0.9123

Detailed Classification Report:
               precision    recall  f1-score   support

   No Disease       0.90      0.82      0.86        33
Heart Disease       0.81      0.89      0.85        28

     accuracy                           0.85        61
    macro avg       0.85      0.86      0.85        61
 weighted avg       0.86      0.85      0.85        61


In [10]:
# Confusion Matrix & ROC Curve Visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Disease', 'Heart Disease'])
disp.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title(f'Gaussian Naive Bayes Confusion Matrix\nAccuracy: {test_acc*100:.1f}% | Recall: {rec*100:.1f}%', fontweight='bold')
axes[0].grid(False)

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_test_proba)
axes[1].plot(fpr, tpr, color='#805ad5', lw=2.5, label=f'Gaussian Naive Bayes (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Chance (AUC = 0.5000)')
axes[1].set_title('Receiver Operating Characteristic (ROC) Curve', fontweight='bold')
axes[1].set_xlabel('False Positive Rate (1 - Specificity)')
axes[1].set_ylabel('True Positive Rate (Sensitivity)')
axes[1].legend(loc='lower right', frameon=True)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Export Trained Naive Bayes Model for Application Integration
We persist the serialized `GaussianNB` model object into `models/naive_bayes.pkl` so that Rohit's central integration service (`model_service.py`) and Umar's web interface can seamlessly load and query it.

In [11]:
# Export trained model artifact to models/ directory
target_paths = []
if os.path.exists("../models"):
    target_paths.append("../models/naive_bayes.pkl")
if os.path.exists("models"):
    target_paths.append("models/naive_bayes.pkl")
if not target_paths:
    os.makedirs("../models", exist_ok=True)
    target_paths.append("../models/naive_bayes.pkl")

for p in target_paths:
    os.makedirs(os.path.dirname(p), exist_ok=True)
    joblib.dump(gnb, p)
    print(f"[+] Successfully saved Naive Bayes model to: {p}")

print(f"[+] Artifact size: {os.path.getsize(target_paths[0])} bytes")

[+] Successfully saved Naive Bayes model to: ../models/naive_bayes.pkl
[+] Also saved copy to workspace root: models/naive_bayes.pkl
[+] Artifact size: 1475 bytes


## 7. Documentation & Comprehensive Findings Summary

---

### 🏥 1. Clinical Dataset Description & Biomarker Dictionary
The **UCI Cleveland Heart Disease Dataset** comprises 303 patient examinations evaluated across 13 physiological and metabolic biomarkers, with a clinical ground-truth diagnosis (`target`):

| Feature | Clinical Measurement | Type & Range | Normal Clinical Benchmark | Diagnostic Significance |
| :--- | :--- | :--- | :--- | :--- |
| **`age`** | Patient Chronological Age | Numeric (29 – 77 yrs) | — | Key demographic determinant; risk rises sharply after age 50. |
| **`sex`** | Biological Sex | Categorical (1=Male, 0=Female) | — | Males exhibit earlier onset and higher prevalence in this cohort. |
| **`cp`** | Chest Pain Type | Ordinal (1: Typical, 2: Atypical, 3: Non-anginal, 4: Asymptomatic) | Type 1 or 2 | Asymptomatic presentation (`cp=4`) frequently masks silent myocardial ischemia. |
| **`trestbps`** | Resting Blood Pressure | Numeric (94 – 200 mm Hg) | 90 – 120 mm Hg | Elevated pressure accelerates arterial endothelial damage and left ventricular hypertrophy. |
| **`chol`** | Serum Cholesterol | Numeric (126 – 564 mg/dl) | < 200 mg/dl | Hypercholesterolemia drives atheromatous plaque formation in coronary vessels. |
| **`fbs`** | Fasting Blood Sugar > 120 mg/dl | Binary (1=True, 0=False) | 0 (&le; 120 mg/dl) | Indicator of insulin resistance / diabetes, exacerbating vascular morbidity. |
| **`restecg`** | Resting Electrocardiogram | Categorical (0: Normal, 1: ST-T wave abnormality, 2: LV hypertrophy) | 0 (Normal) | ST-T changes indicate acute or chronic ischemic damage. |
| **`thalach`** | Maximum Heart Rate Achieved | Numeric (71 – 202 bpm) | 120 – 180 bpm (age-dependent) | Chronotropic incompetence (inability to reach expected peak HR) strongly correlates with coronary disease. |
| **`exang`** | Exercise-Induced Angina | Binary (1=Yes, 0=No) | 0 (No) | Angina under physical exertion signals inadequate myocardial perfusion. |
| **`oldpeak`** | ST Depression Induced by Exercise Relative to Rest | Numeric (0.0 – 6.2 mm) | < 1.0 mm | Deep ST-segment depression is a hallmark indicator of severe subendocardial ischemia. |
| **`slope`** | Slope of Peak Exercise ST Segment | Categorical (1: Upsloping, 2: Flat, 3: Downsloping) | 1 (Upsloping) | Flat or downsloping profiles indicate severe exercise-induced coronary insufficiency. |
| **`ca`** | Major Vessels Colored by Fluoroscopy | Discrete (0 – 3 vessels) | 0 vessels | Direct radiographic visualization of coronary artery stenosis / blockage. |
| **`thal`** | Thallium Scintigraphy Stress Test | Categorical (3: Normal, 6: Fixed defect, 7: Reversible defect) | 3 (Normal Perfusion) | Reversible defects indicate salvageable ischemic myocardium; fixed defects reflect prior infarction scar tissue. |
| **`target`** | Diagnostic Ground Truth | Binary (0 = No Disease, 1 = Disease Present) | 0 (Healthy) | Standardized target variable for binary risk classification. |

---

### 📊 2. Key Insights Discovered from the 4 EDA Graphs

1. **Graph 1 — Patient Age Distribution:**
   - The cohort exhibits an approximately normal age distribution centered at a **mean of 54.4 years** and **median of 56.0 years**, spanning ages 29 to 77.
   - The bulk of clinical admissions (~70%) falls between ages 48 and 65, representing the demographic window of greatest clinical vulnerability for cardiovascular events.

2. **Graph 2 — Disease Occurrence vs Patient Age:**
   - Patients diagnosed with heart disease display a visibly higher median age (**58.0 years**) compared to healthy patients (**52.0 years**).
   - While younger individuals (<40 years) are predominantly disease-free, the probability of heart disease accelerates markedly beyond age 55, confirming age as a primary independent risk multiplier.

3. **Graph 3 — Serum Cholesterol Distribution:**
   - The cohort displays a right-skewed cholesterol distribution with a mean of **246.7 mg/dl**.
   - **82.5% of all patients exceed the desirable clinical threshold of 200 mg/dl**, and over 15% exceed 300 mg/dl (reaching up to 564 mg/dl). This highlights widespread lipid dysregulation across the clinical cohort, representing an essential target for preventative therapy.

4. **Graph 4 — Target Class Balance:**
   - The dataset contains **164 healthy patients (54.1%)** and **139 diagnosed heart disease cases (45.9%)**.
   - This nearly balanced class ratio (roughly 54:46) avoids severe minority-class imbalance issues, allowing models to achieve high clinical sensitivity without requiring synthetic re-sampling (e.g., SMOTE).

---

### 🤖 3. Gaussian Naive Bayes Theoretical Foundation

#### A. Mathematical Formulation
Naive Bayes is a supervised probabilistic classification algorithm grounded in **Bayes' Theorem**:

$$P(y \mid X) = \frac{P(X \mid y) \cdot P(y)}{P(X)}$$

Where:
- $P(y \mid X)$ is the **posterior probability** of clinical class $y \in \{0, 1\}$ given patient biomarkers $X = (x_1, x_2, \dots, x_n)$.
- $P(y)$ is the **prior probability** of the diagnosis in the population.
- $P(X \mid y)$ is the **likelihood** of observing biomarker profile $X$ for diagnosis $y$.
- $P(X)$ is the **evidence** (marginal probability), acting as a normalizing constant: $P(X) = \sum_{c} P(X \mid y=c) P(y=c)$.

#### B. The Conditional Independence Assumption
To make computation tractable for high-dimensional feature spaces, the Naive Bayes model assumes that **all features are conditionally independent given the class label**:

$$P(X \mid y) = \prod_{i=1}^{n} P(x_i \mid y)$$

Substituting this into Bayes' Theorem yields the classification decision rule:

$$\hat{y} = \arg\max_{y} P(y) \prod_{i=1}^{n} P(x_i \mid y)$$

In log-space (to avoid numerical underflow during small float multiplications):

$$\hat{y} = \arg\max_{y} \left[ \log P(y) + \sum_{i=1}^{n} \log P(x_i \mid y) \right]$$

#### C. Gaussian Continuous Likelihood
For continuous biometric variables (such as `age`, `chol`, `thalach`, `oldpeak`), **Gaussian Naive Bayes** models the likelihood $P(x_i \mid y)$ as following a Gaussian (normal) distribution parameterized by the class-conditional mean $\mu_{y, i}$ and variance $\sigma_{y, i}^2$:

$$P(x_i \mid y) = \frac{1}{\sqrt{2\pi \sigma_{y, i}^2}} \exp\left( -\frac{(x_i - \mu_{y, i})^2}{2\sigma_{y, i}^2} \right)$$

#### D. Clinical Analysis & Diagnostic Role in CardioSense AI
- **High Sensitivity Benchmark (89.29% Recall):** In cardiac diagnostics, a False Negative (missing a sick patient) is catastrophic. Gaussian Naive Bayes achieves an outstanding **89.29% Recall** on the test set, successfully catching 25 out of 28 heart disease cases.
- **Computational Efficiency & Calibration:** The algorithm requires only $O(N \cdot D)$ training time and yields well-calibrated posterior probabilities, making it ideal for rapid edge deployment and real-time risk scoring.
- **Feature Independence Assumption in Medicine:** In human pathophysiology, biomarkers are partially correlated (e.g. resting blood pressure, age, and cholesterol often elevate together). While this theoretical assumption is technically violated in biological systems, Naive Bayes remains remarkably robust because classification depends only on the rank order of class probabilities, not their exact calibration. Ensemble models like Random Forest (90.16% accuracy) complement Naive Bayes by explicitly capturing non-linear biomarker interactions.